# E4 trên Kaggle — chỉ nạp final checkpoint, suy luận, chấm và xuất báo cáo

Notebook **không chứa vòng huấn luyện**. Không cần checkpoint final E3.
Trong Kaggle bật GPU/Internet, Add Input dataset đường hư hại và checkpoint
final **E4** của bạn (Output notebook trước, dataset hoặc ZIP).
Checkpoint cần adapter, processor, `extra_trainable.pt` và `run.json`.

Mặc định tự tìm checkpoint và suy ra biến/giá trị E4 từ run.json. Có thể đặt
`FINAL_CHECKPOINT` để chạy một checkpoint cụ thể, hoặc `E4_CHECKPOINTS` để
chạy nhiều nhánh. Ví dụ rank 4/64: thêm cả hai final vào Input để báo cáo đủ
ablation. Nếu chỉ có rank 4, vẫn xuất đầy đủ kết quả rank 4 và ghi rõ còn thiếu
rank 64; không tự train nhánh thiếu hoặc kết luận so sánh chưa có bằng chứng.

Output: JSONL prediction, `report/evidence/e4_*.json`, sáu ảnh GT/prediction,
`report/E4_REPORT.md`, REPORT.md cập nhật bảng E4/Câu 10, và
`/kaggle/working/e4_results.zip`. Chế độ val không cập nhật báo cáo test.


## 0. Cấu hình đường dẫn Kaggle
Để None/{} nếu Input chỉ có checkpoint E4 cần chấm. Nếu có nhiều ablation,
chọn `E4_VARIABLE="rank"` hoặc đường dẫn rõ ràng. Tất cả đầu ra ở working;
checkpoint và dữ liệu Input giữ nguyên.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import zipfile

BASE_SETUP = "E3"  # baseline trong báo cáo; chỉ cần run/evidence đã nhúng
E4_VARIABLE = None  # tự suy ra từ run.json; hoặc "rank" / "data_size" / "coord_scale"
E4_SPLIT = "test"         # val để kiểm tra; val không cập nhật REPORT.md chính
FINAL_CHECKPOINT = None   # ví dụ Path("/kaggle/input/my-e4-checkpoint/work/e4_rank_4/final")
CHECKPOINT_ZIP = None     # ví dụ Path("/kaggle/input/my-e4-checkpoint/e4-final.zip")
DATASET_ROOT = None       # prepared data/ hoặc thư mục có images/ + labels-YOLO/
BASE_MODEL_PATH = None    # model đầy đủ đã thêm vào Input; None tải từ Hugging Face
E4_CHECKPOINTS = {}       # ví dụ {"e4_rank_4": Path("/kaggle/input/e4-r4/final"), "e4_rank_64": Path("/kaggle/input/e4-r64/final")}
INFERENCE_BATCH_SIZE = 4

INPUT_DIR = Path("/kaggle/input")
WORKSPACE = Path("/kaggle/working/day10-e4")
WORKSPACE.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP = WORKSPACE.parent / "e4_results.zip"
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))


In [ ]:
import base64
import gzip
ASSET_BUNDLE = (
    '...'
    'pfw8Re+Lm89+/f8DpsQlVDOFAwA='
)

SOURCE_BUNDLE = json.loads(gzip.decompress(base64.b64decode(ASSET_BUNDLE)).decode("utf-8"))

# Chỉ tạo file chưa có; khi chạy lại không ghi đè báo cáo đã hoàn thiện.
for relative, text in SOURCE_BUNDLE.items():
    destination = WORKSPACE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        destination.write_text(text, encoding="utf-8")
    elif relative.startswith("scripts/"):
        if destination.read_text(encoding="utf-8") != text:
            raise RuntimeError(f"Script của đề đã khác bản gốc: {destination}")
print("Workspace:", WORKSPACE)
print("Scripts của đề được giữ nguyên; báo cáo/evidence cũ được bảo toàn.")


In [ ]:
!nvidia-smi -L
!pip -q install -r /kaggle/working/day10-e4/requirements-colab.txt


## 1. Dữ liệu, kiểm tra final E4 và các hàm suy luận
Loại `model.visual` khỏi NF4, nạp lại projector fp32. Checkpoint được kiểm tra
đủ file và đủ số bước trước khi dùng GPU; chỉ run.json không đủ để suy luận.


In [ ]:
import gc
import math
import random
import re
import subprocess
import time
from collections import Counter
from copy import deepcopy
from functools import partial

import torch
import yaml
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Markdown, FileLink
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import PeftModel
from scripts.labcommon import CLASSES, read_jsonl
from scripts.score import parse_output, iou_xyxy, _gt_boxes, _to_norm1000, _match

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
WORK_DIR = WORKSPACE / "work"
EVIDENCE_DIR = WORKSPACE / "report/evidence"
VALIDATION_DIR = WORKSPACE / "report/validation"
DRIVE_BACKUP_DIR = None  # Kaggle: xuất checkpoint trong Output nếu cần
LOADED_FINAL_CHECKPOINTS = set()


In [ ]:
def require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError("Cần GPU CUDA. Trên Colab hãy chọn runtime T4.")

def release_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_processor(model_id=MODEL_ID):
    processor = AutoProcessor.from_pretrained(model_id)
    processor.tokenizer.padding_side = "left"
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    return processor

def verify_base_precision(model, verbose=True):
    import bitsandbytes as bnb

    # Kiểm tra cả lớp và kiểu Parameter: Params4bit có thể vẫn có dtype float
    # trước khi được pack, nên chỉ kiểm tra dtype là chưa đủ.
    quantized_types = (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt)
    quantized_params = (bnb.nn.Params4bit, bnb.nn.Int8Params)
    protected = [(name, module) for name, module in model.named_modules()
                 if "visual" in name.split(".") or name.split(".")[-1] == "lm_head"]
    if not any("visual" in name.split(".") for name, _ in protected):
        raise RuntimeError("Không tìm thấy ViT trong model; kiểm tra cây named_modules().")
    bad_modules = [name for name, module in protected if isinstance(module, quantized_types)]
    bad_params = []
    wrong_dtype = []
    for name, module in protected:
        for key, param in module.named_parameters(recurse=False):
            full_name = f"{name}.{key}"
            if isinstance(param, quantized_params):
                bad_params.append(full_name)
            elif param.dtype != torch.float16:
                wrong_dtype.append(f"{full_name}: {param.dtype}")
    if bad_modules or bad_params:
        examples = (bad_modules + bad_params)[:5]
        raise RuntimeError(
            "ViT/lm_head vẫn bị lượng tử hoá: " + ", ".join(examples)
            + ". Hãy nạp lại model với llm_int8_skip_modules=['model.visual', 'lm_head'] "
            "ở ô định nghĩa hàm phần 3. Không dùng .float() để sửa Params4bit."
        )
    if wrong_dtype:
        raise RuntimeError("ViT/lm_head phải là fp16 trước khi setup: " + ", ".join(wrong_dtype[:5]))
    llm_4bit = [name for name, module in model.named_modules()
                if "language_model" in name.split(".") and isinstance(module, bnb.nn.Linear4bit)]
    if not llm_4bit:
        raise RuntimeError("LLM chưa có Linear4bit; kiểm tra cấu hình NF4 của model gốc.")
    if verbose:
        print(f"Kiểm tra lượng tử hoá: LLM có {len(llm_4bit)} Linear4bit; "
              "ViT/projector/lm_head không lượng tử hoá, trọng số fp16.")

def load_base_model(model_id=MODEL_ID):
    require_cuda()
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        # Transformers 5.19 so khớp theo đường dẫn đầy đủ/prefix.
        # "visual" không khớp "model.visual.blocks...".
        llm_int8_skip_modules=["model.visual", "lm_head"],
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        BASE_MODEL_PATH or model_id, quantization_config=quantization, dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa",
    )
    verify_base_precision(model)
    return model

def user_message(image, prompt):
    return {"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}

def predict_to_jsonl(model, processor, rows, out_path, coord="norm1000",
                     batch_size=4, max_new_tokens=256):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    model.config.use_cache = True
    model.config.text_config.use_cache = True
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    with out_path.open("w", encoding="utf-8") as stream, torch.inference_mode():
        for start in range(0, len(rows), batch_size):
            chunk = rows[start:start + batch_size]
            messages = [[user_message(open_image(row), make_prompt(
                coord, row["width"], row["height"]))] for row in chunk]
            inputs = processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True,
            )
            inputs.pop("token_type_ids", None)
            inputs = inputs.to(model.device)
            # Autocast cũng cần khi projector đã được nạp ở float32 (E2/E3).
            with torch.autocast("cuda", dtype=torch.float16):
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                    pad_token_id=processor.tokenizer.pad_token_id,
                )
            raw = processor.batch_decode(
                generated[:, inputs["input_ids"].shape[1]:],
                skip_special_tokens=True, clean_up_tokenization_spaces=False,
            )
            for row, text in zip(chunk, raw):
                stream.write(json.dumps({"image_id": row["image_id"], "raw": text},
                                        ensure_ascii=False) + "\n")
            stream.flush()
            print(f"{out_path.name}: {min(start + batch_size, len(rows))}/{len(rows)} ảnh")
            del inputs, generated
    return out_path

def yolo_to_box(label, yolo, coord="norm1000", width=None, height=None):
    if label not in CLASSES:
        raise ValueError(f"Lớp không hợp lệ: {label}")
    if coord == "norm1000":
        sx = sy = 1000
    elif coord == "pixel" and width and height:
        sx, sy = width, height
    else:
        raise ValueError("coord phải là norm1000 hoặc pixel (có width/height).")
    xc, yc, w, h = yolo
    xyxy = ((xc - w / 2) * sx, (yc - h / 2) * sy,
            (xc + w / 2) * sx, (yc + h / 2) * sy)
    box = [int(max(0, min(limit, round(value))))
           for value, limit in zip(xyxy, (sx, sy, sx, sy))]
    return {"label": label, "bbox_2d": box}

def objects_to_answer(objects, coord="norm1000", width=None, height=None):
    boxes = [yolo_to_box(o["label"], o["yolo"], coord, width, height)
             for o in objects]
    boxes.sort(key=lambda o: (o["bbox_2d"][1], o["bbox_2d"][0]))
    return json.dumps(boxes, ensure_ascii=False, separators=(",", ":"))

def make_prompt(coord="norm1000", width=None, height=None):
    if coord == "norm1000":
        scale = "integer coordinates on a 0–1000 scale, normalized by image width and height"
    elif coord == "pixel" and width and height:
        scale = (f"integer pixel coordinates in the original {width}×{height} image, "
                 f"x in [0,{width}] and y in [0,{height}]")
    else:
        raise ValueError("Thang toạ độ không hợp lệ hoặc thiếu kích thước ảnh.")
    return ("Detect every pothole, crack and manhole in this road image. "
            "Return ONLY a JSON list, without markdown or explanation. "
            'Each item must have the format {"label":"pothole","bbox_2d":[x1,y1,x2,y2]}. '
            "The only allowed labels are pothole, crack, manhole. "
            f"Use {scale}; x1 < x2 and y1 < y2. "
            "Order boxes top to bottom, then left to right. Return [] if no objects exist.")

def experiment_config(setup):
    if setup not in ("E1", "E2", "E3"):
        raise ValueError("Chọn E1, E2 hoặc E3.")
    cfg = yaml.safe_load(Path(f"configs/{setup.lower()}.yaml").read_text())
    cfg.update(setup=setup, coord="norm1000", train_fraction=1.0,
               max_new_tokens=256, warmup_ratio=0.05, weight_decay=0.0)
    return cfg

def is_projector(name):
    return "visual.merger." in name or "visual.deepstack_merger_list." in name


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def config_sha256(config):
    return hashlib.sha256(json.dumps(config, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def extract_input_zip(path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"Đường dẫn ZIP không hợp lệ: {item.filename}")
        archive.extractall(destination)
    return destination

def find_dataset(input_dir, explicit=None):
    roots = [Path(explicit)] if explicit else [Path(input_dir)]
    prepared, raw = [], []
    for root in roots:
        if (root / "splits/meta.json").is_file() and (root / "images").is_dir():
            prepared.append(root)
        if (root / "labels-YOLO").is_dir() and (root / "images").is_dir():
            raw.append(root)
        for path in root.rglob("meta.json"):
            if path.parent.name == "splits" and (path.parent.parent / "images").is_dir():
                prepared.append(path.parent.parent)
        for path in root.rglob("labels-YOLO"):
            if path.is_dir() and (path.parent / "images").is_dir():
                raw.append(path.parent)
    candidates = sorted(set(prepared or raw))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Tìm thấy {len(candidates)} dataset phù hợp: {candidates}. "
            "Hãy Add Input dataset và đặt DATASET_ROOT đúng thư mục.")
    selected = candidates[0]
    if (selected / "splits/meta.json").is_file():
        return selected
    destination = WORKSPACE / "data"
    # Gọi nguyên script prepare_data.py, dùng --source cho dataset đã mount.
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--source", str(selected),
                    "--out", str(destination)], check=True)
    return destination

def normalize_run(run):
    run = deepcopy(run)
    original = run["config"]
    setup = original.get("setup", run.get("setup", run["name"]))
    config = experiment_config(setup)
    config.update(original)
    config["lora"] = {**experiment_config(setup)["lora"], **original["lora"]}
    config["name"], config["setup"] = run["name"], setup
    run["config"] = config
    return run

def inspect_checkpoint(checkpoint):
    root = Path(checkpoint)
    adapter = root if (root / "adapter_config.json").is_file() else root / "adapter"
    processor = root if (root / "tokenizer_config.json").is_file() else root / "processor"
    run_path = next((path for path in (root / "run.json", root.parent / "run.json")
                     if path.is_file()), None)
    weights = [path for path in (adapter / "adapter_model.safetensors", adapter / "adapter_model.bin")
               if path.is_file()]
    missing = []
    if run_path is None: missing.append(str(root / "run.json"))
    if not (adapter / "adapter_config.json").is_file(): missing.append(str(adapter / "adapter_config.json"))
    if not weights: missing.append(str(adapter / "adapter_model.safetensors"))
    if not (processor / "tokenizer_config.json").is_file(): missing.append(str(processor / "tokenizer_config.json"))
    if not (processor / "tokenizer.json").is_file(): missing.append(str(processor / "tokenizer.json"))
    if not any((processor / name).is_file() for name in ("processor_config.json", "preprocessor_config.json")):
        missing.append(str(processor / "processor_config.json"))
    if missing:
        raise FileNotFoundError("Checkpoint chưa đủ file: " + ", ".join(missing))
    run = normalize_run(json.loads(run_path.read_text()))
    config = run["config"]
    steps = run.get("step", run.get("completed_steps"))
    if steps != config["max_steps"]:
        raise ValueError(f"Cần checkpoint đã train đủ {config['max_steps']} bước, hiện là {steps}.")
    adapter_config = json.loads((adapter / "adapter_config.json").read_text())
    if adapter_config["r"] != config["lora"]["r"] or adapter_config["lora_alpha"] != config["lora"]["alpha"]:
        raise ValueError("Adapter rank/alpha không khớp run.json.")
    extra = root / "extra_trainable.pt"
    if config["train_projector"] and not extra.is_file():
        raise FileNotFoundError(f"Thiếu projector đã huấn luyện: {extra}")
    return {"root": root, "adapter": adapter, "processor": processor,
            "run_path": run_path, "run": run, "weights": weights, "extra": extra}

def checkpoint_signature(spec):
    paths = [spec["run_path"], spec["adapter"] / "adapter_config.json", *spec["weights"],
             spec["processor"] / "tokenizer_config.json", spec["processor"] / "tokenizer.json"]
    if spec["extra"].is_file(): paths.append(spec["extra"])
    paths.extend(path for path in (spec["processor"] / "processor_config.json",
                                  spec["processor"] / "preprocessor_config.json") if path.is_file())
    return config_sha256({str(path.relative_to(spec["root"])) if path.is_relative_to(spec["root"])
                          else path.name: file_sha256(path) for path in paths})

def open_image(sample):
    with Image.open(DATA_DIR / sample["image_path"]) as source:
        image = source.convert("RGB")
    if image.size != (sample["width"], sample["height"]):
        raise ValueError(f"Kích thước ảnh không khớp nhãn: {sample['image_id']}")
    return image


In [ ]:
def semantic_config(config):
    return {key: deepcopy(value) for key, value in config.items()
            if key not in ("name", "ablation", "ablation_value", "base_setup")}

def identify_e4(spec):
    run, config = spec["run"], spec["run"]["config"]
    if not run["name"].upper().startswith("E4_"):
        raise ValueError(f"Đây không phải checkpoint E4: {run['name']}")
    variable = config.get("ablation")
    value = config.get("ablation_value")
    options = yaml.safe_load(Path("configs/e4.yaml").read_text())["options"]
    if variable not in options or value not in options[variable]["values"]:
        raise ValueError("run.json phải ghi ablation và ablation_value đúng configs/e4.yaml.")
    actual = config["lora"]["r"] if variable == "rank" else config["train_fraction"] if variable == "data_size" else config["coord"]
    if actual != value or (variable == "rank" and config["lora"]["alpha"] != 2 * value):
        raise ValueError("Biến E4 trong metadata không khớp cấu hình checkpoint.")
    expected_name = f"E4_{variable}_{value}".upper()
    if run["name"] != expected_name:
        raise ValueError(f"Tên run cần là {expected_name}, hiện là {run['name']}.")
    return variable, value

def find_e4_checkpoints(input_dir, explicit=None, archive=None, mapping=None, variable=None):
    specs = []
    if mapping:
        specs = [inspect_checkpoint(path) for path in mapping.values()]
    elif explicit:
        specs = [inspect_checkpoint(explicit)]
    else:
        roots = [Path(input_dir)]
        if archive:
            roots = [extract_input_zip(archive, WORKSPACE / "unpacked/e4")]
        def collect(search_roots):
            found = []
            for root in search_roots:
                for path in root.rglob("adapter_config.json"):
                    checkpoint = path.parent.parent if path.parent.name == "adapter" else path.parent
                    try:
                        spec = inspect_checkpoint(checkpoint)
                        v, _ = identify_e4(spec)
                        if variable is None or v == variable: found.append(spec)
                    except (KeyError, ValueError, FileNotFoundError):
                        continue
            return found
        specs = collect(roots)
        if not specs and not archive:
            unpacked = []
            for index, path in enumerate(Path(input_dir).rglob("*.zip")):
                with zipfile.ZipFile(path) as z:
                    relevant = any(name.endswith("adapter_config.json") for name in z.namelist())
                if relevant: unpacked.append(extract_input_zip(path, WORKSPACE / "unpacked" / f"e4-{index}"))
            specs = collect(unpacked)
    unique = {str(spec["root"].resolve()): spec for spec in specs}
    if not unique:
        raise FileNotFoundError("Không tìm được final E4 đủ file. Đặt FINAL_CHECKPOINT hoặc CHECKPOINT_ZIP trong /kaggle/input.")
    variables = {identify_e4(spec)[0] for spec in unique.values()}
    if variable is not None and variables != {variable}:
        raise ValueError(f"Checkpoint không thuộc E4_VARIABLE={variable}: {variables}")
    if len(variables) != 1:
        raise ValueError(f"Input có nhiều biến E4 {variables}; hãy chọn E4_VARIABLE hoặc FINAL_CHECKPOINT.")
    grouped = {}
    for spec in unique.values():
        v, value = identify_e4(spec)
        grouped.setdefault(str(value), []).append(spec)
    selected = []
    for value, candidates in grouped.items():
        finals = [spec for spec in candidates if spec["root"].name == "final"]
        candidates = finals or candidates
        if len(candidates) != 1:
            raise ValueError(f"Có nhiều final cho giá trị {value}; dùng E4_CHECKPOINTS chọn đường dẫn.")
        selected.append(candidates[0])
    return selected

def verify_e4_against_baseline(spec, base_run):
    variable, _ = identify_e4(spec)
    candidate = semantic_config(spec["run"]["config"])
    baseline = semantic_config(base_run["config"])
    if variable == "rank":
        candidate["lora"]["r"] = baseline["lora"]["r"]
        candidate["lora"]["alpha"] = baseline["lora"]["alpha"]
    elif variable == "data_size": candidate["train_fraction"] = baseline["train_fraction"]
    else: candidate["coord"] = baseline["coord"]
    if candidate != baseline:
        raise ValueError("Checkpoint E4 thay đổi cấu hình ngoài biến đã khai báo so với run baseline.")


In [ ]:
DATA_DIR = find_dataset(INPUT_DIR, DATASET_ROOT)
dataset_meta = json.loads((DATA_DIR / "splits/meta.json").read_text())
DATASET_VERSION = dataset_meta["dataset_version"]
splits = {name: read_jsonl(DATA_DIR / "splits" / f"{name}.jsonl") for name in ("train", "val", "test")}
for name, rows in splits.items():
    assert len(rows) == dataset_meta["counts"][name]["images"], f"Sai số ảnh split {name}"
    counts = Counter(obj["label"] for row in rows for obj in row["objects"])
    assert dict(counts) == dataset_meta["counts"][name]["boxes"], f"Sai số box split {name}"
    assert len({row["image_id"] for row in rows}) == len(rows), f"Trùng image_id split {name}"
    missing_images = [row["image_path"] for row in rows if not (DATA_DIR / row["image_path"]).is_file()]
    if missing_images: raise FileNotFoundError(f"Thiếu ảnh: {missing_images[:5]}")
sessions = {name: {row["session"] for row in rows} for name, rows in splits.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not sessions[a] & sessions[b], f"Trùng session: {a}/{b}"
E4_FINALS = find_e4_checkpoints(INPUT_DIR, FINAL_CHECKPOINT, CHECKPOINT_ZIP, E4_CHECKPOINTS, E4_VARIABLE)
E4_VARIABLE = identify_e4(E4_FINALS[0])[0]
base_setups = {spec["run"]["config"].get("base_setup", spec["run"]["config"]["setup"]) for spec in E4_FINALS}
if len(base_setups) != 1: raise ValueError("Các checkpoint E4 có baseline khác nhau.")
BASE_SETUP = next(iter(base_setups))
BASE_RUN = normalize_run(json.loads((WORK_DIR / BASE_SETUP.lower() / "run.json").read_text()))
for spec in E4_FINALS:
    if spec["run"].get("dataset_version", DATASET_VERSION) != DATASET_VERSION:
        raise ValueError("Checkpoint và dataset khác phiên bản.")
    verify_e4_against_baseline(spec, BASE_RUN)
    print("E4 final:", spec["root"], "|", identify_e4(spec))
print("Dataset:", DATA_DIR, DATASET_VERSION, {name: len(rows) for name, rows in splits.items()})
print("Chỉ evaluate; không có vòng huấn luyện trong notebook.")


## 2. Nạp final E4 và chấm kết quả
`evaluate_experiment` nạp adapter/projector, sinh prediction và gọi score.py.
Evidence cùng fingerprint đã có sẽ được tái dùng. Không tải final E3 hoặc
đánh giá lại E1–E3. Nếu chỉ có một nhánh, vẫn xuất kết quả nhánh đó.


In [ ]:
def load_trained_model(checkpoint):
    spec = inspect_checkpoint(checkpoint)
    model = PeftModel.from_pretrained(load_base_model(spec["run"]["config"]["model_id"]),
                                      spec["adapter"], is_trainable=False)
    if spec["run"]["config"]["train_projector"]:
        extra = torch.load(spec["extra"], map_location="cpu", weights_only=True)
        params = dict(model.named_parameters())
        expected = {name for name in params if is_projector(name)}
        if set(extra) != expected:
            raise ValueError("Checkpoint projector thiếu/thừa tên tham số.")
        with torch.no_grad():
            for name, tensor in extra.items():
                if params[name].shape != tensor.shape or not tensor.is_floating_point():
                    raise ValueError(f"Projector shape/dtype không hợp lệ: {name}")
                params[name].data = params[name].data.to(torch.float32)
                params[name].copy_(tensor.to(params[name].device, dtype=torch.float32))
        print(f"Đã nạp {sum(t.numel() for t in extra.values()):,} tham số projector fp32.")
    processor = load_processor(spec["processor"])
    LOADED_FINAL_CHECKPOINTS.add(str(spec["root"].resolve()))
    return model, processor, spec["run"]

def validate_score(score, run, split, name):
    config = run["config"]
    for key, expected in {"name": name, "split": split, "coord": config["coord"],
                          "dataset_version": DATASET_VERSION, "score_version": 1,
                          "n_images": len(splits[split]), "missing_predictions": 0}.items():
        if score.get(key) != expected:
            raise ValueError(f"Evidence khác {key}: {score.get(key)} != {expected}")
    for key in ("trainable_params", "peak_vram_gb", "train_minutes"):
        if score["meta"].get(key) != run[key]:
            raise ValueError(f"Evidence {name} không khớp run: {key}")

def evaluation_paths(name, split):
    if split not in ("val", "test"):
        raise ValueError("split phải là val hoặc test.")
    stem = name.lower() + ("_val" if split == "val" else "")
    return WORK_DIR / f"{stem}.jsonl", (EVIDENCE_DIR if split == "test" else VALIDATION_DIR) / f"{stem}.json"

def evaluate_experiment(checkpoint, split="test"):
    spec = inspect_checkpoint(checkpoint)
    run, config = spec["run"], spec["run"]["config"]
    name = run["name"]
    pred_path, evidence_path = evaluation_paths(name, split)
    signature = checkpoint_signature(spec)
    config_hash = config_sha256(config)
    if evidence_path.exists():
        score = json.loads(evidence_path.read_text())
        validate_score(score, run, split, name)
        if name.startswith("E4"):
            if score["meta"].get("config_sha256") != config_hash or score["meta"].get("checkpoint_sha256") != signature:
                raise ValueError(f"Evidence E4 thuộc config/checkpoint khác: {evidence_path}")
            if pred_path.exists() and score["meta"].get("pred_sha256") != file_sha256(pred_path):
                raise ValueError("JSONL prediction đã thay đổi sau khi chấm.")
        print("Tái dùng evidence:", evidence_path)
        return score
    model = None
    try:
        model, processor, run = load_trained_model(spec["root"])
        temporary = pred_path.with_suffix(".inprogress.jsonl")
        predict_to_jsonl(model, processor, splits[split], temporary, coord=config["coord"],
                         batch_size=INFERENCE_BATCH_SIZE, max_new_tokens=config["max_new_tokens"])
        rows = read_jsonl(temporary)
        if len(rows) != len(splits[split]) or {row["image_id"] for row in rows} != {row["image_id"] for row in splits[split]}:
            raise ValueError("Prediction thiếu/trùng image_id.")
        temporary.replace(pred_path)
    finally:
        del model
        release_cuda()
    evidence_path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {key: run[key] for key in ("trainable_params", "peak_vram_gb", "train_minutes")}
    metadata.update({key: config[key] for key in ("setup", "seed", "train_fraction")})
    metadata.update(lora_r=config["lora"]["r"], config_sha256=config_hash,
                    checkpoint_sha256=signature, pred_sha256=file_sha256(pred_path))
    metadata.update({key: config[key] for key in ("ablation", "ablation_value", "base_setup") if key in config})
    command = [sys.executable, "scripts/score.py", "--pred", str(pred_path), "--name", name,
               "--out", str(evidence_path), "--split", split, "--splits-dir", str(DATA_DIR / "splits"),
               "--coord", config["coord"]]
    for key, value in metadata.items(): command += ["--meta", f"{key}={value}"]
    subprocess.run(command, check=True)
    score = json.loads(evidence_path.read_text())
    validate_score(score, run, split, name)
    return score


In [ ]:
def run_e4_evaluation(specs, split="test"):
    results = {}
    for spec in specs:
        variable, value = identify_e4(spec)
        run = spec["run"]
        score = evaluate_experiment(spec["root"], split)
        prediction, evidence = evaluation_paths(run["name"], split)
        copy_to = WORK_DIR / run["name"].lower() / "run.json"
        copy_to.parent.mkdir(parents=True, exist_ok=True)
        if copy_to.resolve() != spec["run_path"].resolve(): shutil.copy2(spec["run_path"], copy_to)
        results[str(value)] = {"score": score, "run": run, "checkpoint": str(spec["root"]),
                               "prediction": str(prediction), "evidence": str(evidence),
                               "config": run["config"], "reused": False}
        print(run["name"], "F1:", score["micro"]["f1"], "CI95:", score["f1_ci95"])
    # Baseline tương đương 100%/norm1000 có thể tái dùng bằng evidence và run,
    # không cần trọng số baseline. Rank 4/64 không tương đương rank 16.
    options = yaml.safe_load(Path("configs/e4.yaml").read_text())["options"][E4_VARIABLE]["values"]
    baseline_value = BASE_RUN["config"]["train_fraction"] if E4_VARIABLE == "data_size" else BASE_RUN["config"]["coord"] if E4_VARIABLE == "coord_scale" else None
    if baseline_value in options and str(baseline_value) not in results:
        _, baseline_path = evaluation_paths(BASE_SETUP, split)
        if baseline_path.is_file():
            score = json.loads(baseline_path.read_text())
            validate_score(score, BASE_RUN, split, BASE_SETUP)
            cfg = deepcopy(BASE_RUN["config"])
            cfg.update(name=f"E4_{E4_VARIABLE}_{baseline_value}".upper(), ablation=E4_VARIABLE,
                       ablation_value=baseline_value, base_setup=BASE_SETUP)
            score = deepcopy(score)
            score["name"] = cfg["name"]
            score["meta"].update(ablation=E4_VARIABLE, ablation_value=baseline_value,
                                  base_setup=BASE_SETUP, reused_from=BASE_SETUP,
                                  baseline_evidence_sha256=file_sha256(baseline_path))
            prediction, evidence = evaluation_paths(cfg["name"], split)
            evidence.parent.mkdir(parents=True, exist_ok=True)
            evidence.write_text(json.dumps(score, ensure_ascii=False, indent=2), encoding="utf-8")
            baseline_prediction, _ = evaluation_paths(BASE_SETUP, split)
            results[str(baseline_value)] = {"score": score, "run": BASE_RUN, "checkpoint": None,
                                           "prediction": str(baseline_prediction), "evidence": str(evidence),
                                           "config": cfg, "reused": True}
    return {str(value): results[str(value)] for value in options if str(value) in results}

E4_RESULTS = run_e4_evaluation(E4_FINALS, split=E4_SPLIT)


## 3. Kết quả, ảnh dự đoán và báo cáo tải về
Tự cập nhật bảng E4 và Câu 10 bằng số vừa chấm. Nếu thiếu nhánh thứ hai, báo
cáo nêu rõ nhánh thiếu và không đưa ra kết luận ablation. ZIP không chứa trọng số.


In [ ]:
def prediction_details(rows, prediction_path, coord):
    raw_rows = read_jsonl(prediction_path)
    raw = {item["image_id"]: item["raw"] for item in raw_rows}
    if len(raw_rows) != len(rows) or len(raw) != len(rows) or set(raw) != {row["image_id"] for row in rows}:
        raise ValueError("JSONL không đủ/khớp split để minh họa.")
    details = []
    for row in rows:
        parsed = parse_output(raw[row["image_id"]])
        pred = [] if parsed is None else _to_norm1000(parsed, coord, row["width"], row["height"])
        gt = _gt_boxes(row)
        counts, _ = _match(gt, pred)
        totals = [sum(values[index] for values in counts.values()) for index in range(3)]
        details.append({"row": row, "gt": gt, "pred": pred, "raw": raw[row["image_id"]],
                        "valid": parsed is not None, "tp": totals[0], "fp": totals[1], "fn": totals[2]})
    return details

def prediction_gallery(result, split, destination):
    details = prediction_details(splits[split], result["prediction"], result["config"]["coord"])
    selected, used = [], set()
    for metric in ("tp", "fp", "fn"):
        ordered = sorted(details, key=lambda item: (-item[metric], item["row"]["image_id"]))
        for item in ordered:
            if item["row"]["image_id"] not in used:
                selected.append(item)
                used.add(item["row"]["image_id"])
                if len(selected) >= 2 * (("tp", "fp", "fn").index(metric) + 1): break
    canvas = Image.new("RGB", (1304, 1320), "#142034")
    font = ImageFont.load_default()
    draw = ImageDraw.Draw(canvas)
    draw.text((12, 12), f"{result['score']['name']} {split} | GREEN=GT, RED=PRED | IoU matching >= 0.5", fill="white", font=font)
    summaries = []
    for index, item in enumerate(selected[:6]):
        row = item["row"]
        image = open_image(row)
        painter = ImageDraw.Draw(image)
        for prefix, boxes, color in (("GT", item["gt"], "#70ff8a"), ("P", item["pred"], "#ff6262")):
            for obj in boxes:
                x1, y1, x2, y2 = obj["bbox"]
                box = (x1 / 1000 * image.width, y1 / 1000 * image.height,
                       x2 / 1000 * image.width, y2 / 1000 * image.height)
                painter.rectangle(box, outline=color, width=2)
                painter.text((max(0, box[0]), max(0, box[1] - 12)), prefix + ":" + obj["label"], fill=color, font=font)
        x, y = 12 + index % 2 * 652, 42 + index // 2 * 422
        canvas.paste(image, (x, y))
        draw.text((x, y + 367), row["image_id"], fill="white", font=font)
        draw.text((x, y + 383), f"TP={item['tp']} FP={item['fp']} FN={item['fn']} parse_valid={item['valid']}", fill="white", font=font)
        draw.text((x, y + 399), "session=" + row["session"], fill="white", font=font)
        summaries.append({"image_id": row["image_id"], "session": row["session"],
                          "tp": item["tp"], "fp": item["fp"], "fn": item["fn"],
                          "parse_valid": item["valid"], "raw": item["raw"]})
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(destination)
    return summaries

def score_table_row(name, setup, score):
    meta = score["meta"]
    ci = score["f1_ci95"]
    values = [name, setup, f"{meta['trainable_params']:,}", f"{meta['peak_vram_gb']:.4f}",
              f"{meta['train_minutes']:.4f}", f"{score['micro']['f1']:.4f}",
              f"[{ci[0]:.4f}; {ci[1]:.4f}]", f"{score['fp_per_image']:.4f}"]
    values += [f"{score['per_class'][label]['f1']:.4f}" for label in CLASSES]
    return "| " + " | ".join(values) + " |"

def package_results():
    destination = EXPORT_ZIP
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        candidates = [WORKSPACE / "REPORT.md", *list((WORKSPACE / "report").rglob("*")),
                      *list(WORK_DIR.glob("*.jsonl")), *list(WORK_DIR.glob("e*/run.json")),
                      *list(WORK_DIR.glob("e4_*/final/run.json")), *list(WORK_DIR.glob("e4_*/train_image_ids.json"))]
        for path in sorted(set(candidates)):
            if path.is_file(): archive.write(path, path.relative_to(WORKSPACE))
    return destination


In [ ]:
def export_e4_report(results, split="test"):
    if not results: raise ValueError("Chưa có kết quả E4.")
    options = yaml.safe_load(Path("configs/e4.yaml").read_text())["options"][E4_VARIABLE]["values"]
    missing = [str(value) for value in options if str(value) not in results]
    for result in results.values():
        if result["score"]["split"] != split or result["score"]["dataset_version"] != DATASET_VERSION:
            raise ValueError("Không trộn split/dataset khi báo cáo.")
    text = [f"Đánh giá checkpoint final E4 biến **{E4_VARIABLE}**, baseline **{BASE_SETUP}**, "
            f"dataset `{DATASET_VERSION}`, split **{split}**. Notebook chỉ suy luận/chấm, không huấn luyện lại.", "",
            f"Cấu hình ghi trong run: seed {BASE_RUN['config']['seed']}, lr {BASE_RUN['config']['lr']}, "
            f"batch {BASE_RUN['config']['batch_size']}, grad_accum {BASE_RUN['config']['grad_accum']}, "
            f"{BASE_RUN['config']['max_steps']} bước train. Các cấu hình ngoài biến E4 đã đối chiếu với run baseline."]
    for value, result in results.items():
        score = result["score"]
        meta = score["meta"]
        text += ["", f"Giá trị **{value}**: F1 **{score['micro']['f1']:.4f}**, CI95 **{score['f1_ci95']}**, "
                 f"precision **{score['micro']['precision']:.4f}**, recall **{score['micro']['recall']:.4f}**, "
                 f"TP/FP/FN **{score['micro']['tp']}/{score['micro']['fp']}/{score['micro']['fn']}**; "
                 f"parse_valid_rate **{score['parse_valid_rate']:.4f}**, FP/ảnh **{score['fp_per_image']:.4f}**.",
                 f"Tài nguyên train từ run/evidence: **{meta['trainable_params']:,}** tham số, "
                 f"**{meta['peak_vram_gb']:.4f} GB**, **{meta['train_minutes']:.4f} phút**. "
                 + (f"Tái dùng evidence baseline {BASE_SETUP} cho cấu hình tương đương." if result["reused"] else "Chấm từ checkpoint E4 đã cung cấp.")]
    delta = overlap = None
    if not missing and len(results) == 2:
        a, b = (results[str(value)]["score"] for value in options)
        delta = b["micro"]["f1"] - a["micro"]["f1"]
        overlap = max(a["f1_ci95"][0], b["f1_ci95"][0]) <= min(a["f1_ci95"][1], b["f1_ci95"][1])
        text += ["", f"Chênh F1 ({options[1]} − {options[0]}) = **{delta:+.4f}**, "
                 f"tương đương **{100 * delta:+.2f} điểm phần trăm**.",
                 "Hai CI95 có chồng nhau; chưa đủ kết luận chênh lệch chắc chắn từ điểm ước lượng."
                 if overlap else "Hai CI95 không chồng nhau, hỗ trợ chênh lệch quan sát trong lần chạy này."]
    else:
        text += ["", f"**Chưa đủ hai giá trị theo GUIDE: còn thiếu {', '.join(missing)}.** "
                 "Kết quả trên hoàn thiện cho checkpoint đã có; chưa thể kết luận so sánh ablation. "
                 "Thêm final/evidence nhánh còn lại rồi chạy lại notebook; không tự train nhánh thiếu."]
    text += ["", "Test chuẩn có 311 ảnh, 3 nhóm session. CI bootstrap theo ảnh có thể đánh giá thấp "
             "phụ thuộc giữa frame, chưa phải CI ghép cặp của hiệu F1; mới có một seed. "
             "Không suy rộng một lần chạy thành bảo đảm cho phiên quay mới."]
    answer = "\n".join(text)
    header = "| Mã | Setup | Tham số huấn luyện | VRAM đỉnh (GB) | Phút train | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole |"
    separator = "|---|---|---|---|---|---|---|---|---|---|---|"
    rows = [score_table_row("E4", f"{BASE_SETUP}; {E4_VARIABLE}={value}", result["score"]) for value, result in results.items()]
    body = f"# E4 — Đánh giá final checkpoint ({E4_VARIABLE})\n\n" + "\n".join([header, separator, *rows]) + "\n\n" + answer
    selected = next((result for result in results.values() if Path(result["prediction"]).is_file()), None)
    figures = []
    caption = ""
    if selected:
        figure = WORKSPACE / "report/figures" / f"e4_{E4_VARIABLE}_{split}_predictions.png"
        examples = prediction_gallery(selected, split, figure)
        figure.with_suffix(".json").write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding="utf-8")
        figures.append(str(figure))
        caption = (f"Sáu ảnh của **{selected['score']['name']}**, split **{split}**: xanh là GT, đỏ là prediction. "
                   "TP/FP/FN theo scorer IoU ≥ 0,5, chọn từ JSONL thật theo số TP/FP/FN. Đây là kết quả E4.")
        body += "\n\n" + caption + f"\n\n![GT và prediction]({figure.relative_to(WORKSPACE / 'report')})\n"
    suffix = "" if split == "test" else "_val"
    report_path = WORKSPACE / "report" / f"E4_REPORT{suffix}.md"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(body + "\n", encoding="utf-8")
    if split == "test":
        main_path = WORKSPACE / "REPORT.md"
        main = main_path.read_text(encoding="utf-8")
        lines = main.splitlines()
        insertion = next(i for i, line in enumerate(lines) if line.startswith("| E4 |"))
        lines = [line for line in lines if not line.startswith("| E4 |")]
        lines[insertion:insertion] = rows
        main = "\n".join(lines) + "\n"
        start = main.index("### Câu 10")
        begin = main.index("\n", start) + 1
        end = main.index("\n## Nhìn lại", begin)
        main = main[:begin] + "\n" + answer + "\n" + main[end:]
        main = main.replace("Chưa có E4; dấu “—” biểu thị chưa có số đo.",
                            f"Đã đánh giá E4 {E4_VARIABLE} các giá trị {', '.join(results)}; mức hoàn thiện ablation ghi ở Câu 10.")
        main = main.replace("Chưa có CI E4 hoặc lặp nhiều seed để đánh giá độ ổn định giữa các lần train.",
                            "Đã có CI cho các checkpoint E4 được đánh giá ở Câu 10; chưa lặp nhiều seed.")
        if selected:
            start9 = main.index("### Câu 9")
            end9 = main.index("### Câu 10", start9)
            section = main[start9:end9]
            picture = section.find("Hình dưới lấy")
            if picture >= 0:
                section = section[:picture] + caption + f"\n\n![GT và prediction]({figure.relative_to(WORKSPACE)})\n\n"
                main = main[:start9] + section + main[end9:]
        if LOADED_FINAL_CHECKPOINTS:
            main = main.replace("workspace hiện mới có run log, chưa có trọng số E2/E3 để kiểm tra độc lập nội dung checkpoint và dtype thực tế.",
                                "Notebook E4 đã nạp checkpoint final E4, kiểm tra ViT/lm_head không lượng tử hóa và nạp projector fp32.")
        backup = WORKSPACE / "report/REPORT_before_E4.md"
        if not backup.exists(): shutil.copy2(main_path, backup)
        main_path.write_text(main, encoding="utf-8")
    summary = {"variable": E4_VARIABLE, "baseline": BASE_SETUP, "split": split,
               "dataset_version": DATASET_VERSION, "results": results,
               "missing_values": missing, "ablation_complete": not missing,
               "f1_delta": delta, "ci_overlap": overlap, "figures": figures,
               "evaluated_checkpoints": sorted(LOADED_FINAL_CHECKPOINTS)}
    summary_path = WORKSPACE / "report" / f"e4_summary{suffix}.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Báo cáo:", report_path)
    return report_path, summary_path


In [ ]:
report_path, summary_path = export_e4_report(E4_RESULTS, split=E4_SPLIT)
display(Markdown(report_path.read_text(encoding="utf-8")))
if E4_SPLIT == "test":
    check = subprocess.run([sys.executable, "scripts/check_submission.py"], capture_output=True, text=True)
    print(check.stdout)
    print("Notebook Kaggle không có lịch sử Git; yêu cầu commit được kiểm tra ở repo khi nộp.")
archive = package_results()
print("Tải từ tab Output sau Save Version (Save & Run All):", archive)
display(FileLink(str(archive)))
display(FileLink(str(report_path)))
